# Testing the finetuned model

In [1]:
prompt = """Doctor: Hey, how are you doing today?

Patient: Hello doctor. I am feeling pain on the bottom right in my belly.

Doctor: How long has the pain been there?

Patient: It started yesterday evening and got worse during the night.

Doctor: Can you describe the pain? Is it sharp, dull, cramping, or something else?

Patient: It started as a dull ache, but now it feels sharp when I move or walk.

Doctor: On a scale from 1 to 10, how strong is the pain?

Patient: Around 7 out of 10.

Doctor: Have you noticed any nausea, vomiting, fever, or changes in appetite?

Patient: Yes, I feel nauseous and I did not want breakfast this morning. I also think I have a slight fever.

Doctor: Have you had diarrhea or constipation?

Patient: No diarrhea, but I have not gone to the bathroom since yesterday.

Doctor: Does anything make the pain better or worse?

Patient: Moving makes it worse. Lying still helps a little.

Doctor: Have you experienced this kind of pain before?

Patient: No, never this bad.

Doctor: Do you have any medical conditions or take any medications regularly?

Patient: No major medical conditions. I only take allergy medicine sometimes.

Doctor: Thank you. I would like to examine your abdomen now, especially the lower right side.

Patient: Okay.

Doctor: When I press here, does it hurt?

Patient: Yes, especially when you let go.

Doctor: I understand. Based on your symptoms and the examination, this could be appendicitis. I recommend blood tests and an abdominal scan as soon as possible.

Patient: Is it serious?

Doctor: It can become serious if untreated, but we caught it early. We will arrange further testing immediately.

Patient: Thank you, doctor.

Doctor: You're welcome. We will take good care of you."""

In [13]:
def build_messages(example: str, use_system_prompt: bool = True) -> list[dict]:
    messages = [
        {"role": "system",
        "content": """You are a medical clinical documentation assistant. 
You task is to convert a dialogue between a doctor and patient into a structured clinical note in the following output format:
REASON FOR VISIT:
<Brief summary of why the patient is seeking care>
PATIENT DETAILS AND HISTORY:
<Age, gender, relevant demographics, relevant past medical history, conditions, medications, surgeries, lifestyle factors>
CURRENT STATUS:
<Current symptoms, findings, vitals, clinical observations>
TREATMENTS/ACTIONS:
<Medications prescribed, procedures performed, advice given>
FOLLOW-UP PLAN:
<Next steps, monitoring, referrals, timelines. Follow-up plan should not include "future" details that are mentioned in the note, but rather should infer what the next steps would be based on the found future details.>
"""},
        {"role": "user",   "content": example},
    ]
    if not use_system_prompt:
        messages = [messages[-1]]

    return messages

In [3]:
from vllm import LLM, SamplingParams
import os

In [4]:
SAMPLING = dict(temperature=0.0, max_tokens=1000)
sampling_params = SamplingParams(**SAMPLING)

In [5]:
SLURM_JOB_ACCOUNT = os.getenv("SLURM_JOB_ACCOUNT") #modify
USER = os.getenv("SLURM_JOB_USER") #modify
output_path = f"/scratch/{SLURM_JOB_ACCOUNT}/{USER}/health_case/ft_model"
input_model = "google/medgemma-1.5-4b-it"
model_output_name = f"{input_model}_finetuned"
merged_output_dir = os.path.join(
    output_path,
    f"{model_output_name}_merged"
)

In [6]:
llm = LLM(model=merged_output_dir, tensor_parallel_size=1, dtype="bfloat16")

INFO 05-29 09:51:59 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'disable_log_stats': True, 'model': '/scratch/project_462000131/hmerilai/health_case/ft_model/google/medgemma-1.5-4b-it_finetuned_merged'}
INFO 05-29 09:51:59 [model.py:555] Resolved architecture: Gemma3ForConditionalGeneration
INFO 05-29 09:51:59 [model.py:1680] Using max model len 131072
INFO 05-29 09:51:59 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-29 09:51:59 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 05-29 09:52:00 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])


The tokenizer you are loading from '/scratch/project_462000131/hmerilai/health_case/ft_model/google/medgemma-1.5-4b-it_finetuned_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


INFO 05-29 09:52:01 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-29 09:52:01 [nixl_utils.py:34] NIXL is not available
WARNING 05-29 09:52:01 [nixl_utils.py:44] NIXL agent config is not available


/opt/venv/lib/python3.12/site-packages/tensorizer/utils.py:23: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
The tokenizer you are loading from '/scratch/project_462000131/hmerilai/health_case/ft_model/google/medgemma-1.5-4b-it_finetuned_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading thi

WARNING 05-29 09:52:11 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore pid=65068) INFO 05-29 09:52:20 [core.py:109] Initializing a V1 LLM engine (v0.20.1) with config: model='/scratch/project_462000131/hmerilai/health_case/ft_model/google/medgemma-1.5-4b-it_finetuned_merged', speculative_config=None, tokenizer='/scratch/project_462000131/hmerilai/health_case/ft_model/google/medgemma-1.5-4b-it_finetuned_merged', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=True, quant

...

(EngineCore pid=65068) INFO 05-29 09:52:39 [default_loader.py:384] Loading weights took 8.21 seconds
(EngineCore pid=65068) INFO 05-29 09:52:39 [gpu_model_runner.py:4879] Model loading took 9.91 GiB memory and 8.750501 seconds
(EngineCore pid=65068) INFO 05-29 09:52:40 [gpu_model_runner.py:5820] Encoder cache will be initialized with a budget of 8192 tokens, and profiled with 32 image items of the maximum feature size.
(EngineCore pid=65068) WARNING 05-29 09:52:40 [op.py:241] Priority not set for op rms_norm, using native implementation.
(EngineCore pid=65068) INFO 05-29 09:52:45 [backends.py:1069] Using cache directory: /users/hmerilai/.cache/vllm/torch_compile_cache/6bcbde8880/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=65068) INFO 05-29 09:52:45 [backends.py:1128] Dynamo bytecode transform time: 2.27 s
(EngineCore pid=65068) INFO 05-29 09:52:47 [backends.py:290] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.222 s
(EngineCore pi

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 14.65it/s]
Capturing CUDA graphs (decode, FULL):   0%|          | 0/35 [00:00<?, ?it/s]

(EngineCore pid=65068) WARNING 05-29 09:52:52 [chunked_prefill_paged_decode.py:400] Cannot use ROCm custom paged attention kernel, falling back to Triton implementation.


Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 15.68it/s]


(EngineCore pid=65068) INFO 05-29 09:52:55 [gpu_model_runner.py:6133] Graph capturing finished in 6 secs, took 0.41 GiB
(EngineCore pid=65068) INFO 05-29 09:52:55 [core.py:299] init engine (profile, create kv cache, warmup model) took 15.61 s (compilation: 4.08 s)
(EngineCore pid=65068) INFO 05-29 09:52:55 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])


## Inference with system prompt

In [7]:
outputs = llm.chat(build_messages(prompt), sampling_params, use_tqdm=True)

Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

The tokenizer you are loading from '/scratch/project_462000131/hmerilai/health_case/ft_model/google/medgemma-1.5-4b-it_finetuned_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


INFO 05-29 10:03:23 [hf.py:314] Detected the chat template content format to be 'openai'. You can set `--chat-template-content-format` to override this.


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [8]:
print(outputs[0].outputs[0].text)

**REASON FOR VISIT:**  
The patient presented with acute right lower quadrant abdominal pain that began the previous evening and progressively worsened overnight.

**PATIENT DETAILS AND HISTORY:**  
The patient is a 22‑year‑old female. She reports no chronic medical conditions, regular medications, or prior surgeries. She occasionally uses over‑the‑counter allergy medication. No lifestyle factors were specified.

**CURRENT STATUS:**  
On examination, the right lower quadrant was tender to palpation, with rebound tenderness present. Vital signs were stable, and laboratory studies showed a white blood cell count of 12.5 ×�0.10 ×10⁹/L with a left shift. Imaging revealed a 1.5 cm fluid‑filled appendix with a small amount of free fluid in the right paracolic gutter.

**TREATMENTS/ACTIONS:**  
The patient was admitted to the hospital and received intravenous fluids. She underwent an appendectomy via a midline incision. Post‑operatively she was monitored in the recovery room, discharged home 

## Inference without system prompt

In [15]:
outputs = llm.chat(build_messages(prompt, use_system_prompt=False), sampling_params, use_tqdm=True)

Rendering conversations:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [16]:
print(outputs[0].outputs[0].text)

I am an AI Chatbot and cannot provide medical advice. The scenario you described is a classic presentation of appendicitis.  **If you or someone you know is experiencing similar symptoms, it is crucial to seek immediate medical attention.**  Appendicitis is a medical emergency that requires prompt diagnosis and treatment to prevent serious complications.

Here's a breakdown of the information provided in the dialogue:

*   **Patient's Symptoms:**
    *   Pain in the lower right abdomen.
    *   Pain started yesterday evening and worsened overnight.
    *   Pain initially dull, now sharp with movement/walking.
    *   Pain intensity: ~7/10.
    *   Nausea, loss of appetite (no breakfast).
    *   Possible low-grade fever.
    *   No diarrhea, but no bowel movement since yesterday.
    *   Pain is worse with movement, slightly better when still.
    *   No prior history of similar severe pain.
    *   No chronic medical conditions, only occasional allergy medication.

*   **Doctor's Asse